In [933]:
import pandas as pd
import numpy as np
import re
from urllib.parse import unquote


import sys
sys.path.append("./submission")
import clean

df = pd.read_csv("./data/test_features.csv")
df = df[['origin_station',
 'destination_station',
 'district',
 'encoded_transport',
 'day_of_week',
 'is_holiday',
 'weather_condition',
 'country_code']]

Lucciano: Cleaning Origin and Destination

In [934]:
# Canonical station -> ordered regexes matched against the SQUASHED key
# (lowercased, url-decoded, wrappers stripped, all non-alphanumerics removed).
# Order matters: more specific patterns first. ^x$ pins the short codes.
_STATION_RULES = [
    ("Causeway Bay", r"causewaybay|cwbhub|^cwb$|^cb$"),
    ("Central", r"central|stncentral|censt|^cstation$|^cstn$|^cs$"),
    ("Admiralty", r"admiralt|admxfer|^adm$"),
    ("Sha Tin", r"shatin|shatinterm|^stin$|^stn$|^st$"),
    ("Mong Kok", r"mongkok|mkhub|^mk$"),
    ("Tsim Sha Tsui", r"tsimshatsui|tsteast|^tst$"),
    ("North Point", r"northpoint|northpt|npferry|^np$"),
    ("Tsuen Wan", r"tsuenwan|tswline|^twn$|^tswan$|^tw$"),
    ("Kennedy Town", r"kennedyt|ktownstop|^ktown$|^kt$"),
    ("Wan Chai", r"wanchai|wanchaistop|^wchai$|^wc$"),
]
# Markers/non-stations to skip. unk = unknown placeholder; depot/workshop/audithub
# are internal sites, not passenger stations -> treated as missing.
_NOISE = re.compile(r"^(unk|audithub|depot|workshop)$")


def clean_station(s: pd.Series) -> pd.Series:
    def clean_value(v):
        if not isinstance(v, str):
            return pd.NA
        x = unquote(unquote(v)).strip().lower()  # decode (handles %2520), trim
        for tok in re.split(r"[:|=\-]+", x):
            key = re.sub(r"[^a-z0-9]", "", tok)  # squash: drop spaces/./%20 etc
            if not key or _NOISE.match(key):
                continue
            for canon, pat in _STATION_RULES:
                if re.search(pat, key):
                    return canon
        return pd.NA  # no station token -> missing

    return s.map(clean_value)

df['origin_station'] = clean_station(df['origin_station'])
df['destination_station'] = clean_station(df['destination_station'])

Cleaning District

In [935]:
# Clean district column values of prefix/suffix patterns

def clean_district(df):
    # Strip whitespace
    df['district'] = df['district'].str.strip()
    
    # Remove prefix patterns (e.g., "D123" → "123")
    prefix_pattern = r'(D|zone|hold|district|Legacy|region|Leg|LEG)[^a-zA-Z0-9]+'
    df['district'] = df['district'].str.replace(prefix_pattern, '', regex=True)
    
    # Remove suffix patterns (e.g., "central_zone" → "central")
    suffix_pattern = r'[_|](?:new|core|zone|cluster|sector|OPS)'
    df['district'] = df['district'].str.replace(suffix_pattern, '', regex=True)
    
    # Normalize missing/weird values into NaN
    unknown_values = [
        'Legacy', 'unknown', '??', 'mismatch', 'Null', 'legacy',
        'Internal', 'operations', 'audit', 'eh', 'unk', ''
    ]
    df['district'] = df['district'].str.replace(r'^(operations|audit|unk|eh)\s*$', '', regex=True, case=False)

    df['district'] = df['district'].replace(unknown_values, np.nan)

    return df

df = clean_district(df)

In [936]:
#map district variants into unified category

district_variant_mapping = [
    ("Central and Western", r"cw|cwbhub|^cwb$|^cb$|central_and_western|central and western"),
    ("Tsuen Wan", r"tw|tsw|tsuen_wan|tsuen wan"),
    ("Yau Tsim Mong", r"YTM|ytm|yau tsim mong|yau_tsim_mong"),
    ("Sha Tin", r"ST|sha tin|shatin|st|sha_tin"),
    ("Eastern", r"eastern|east_harbour"), 
    ("Wan Chai", r"wanchai|wanchaistop|^wchai$|^wc$|wan chai|wan_chai")
]

def map_district(value):
    if pd.isna(value):
        return value
    value = str(value).lower().strip()
    for full_name, pattern in district_variant_mapping:
        if pattern and re.search(pattern, value):
            return full_name
    return value  # return original if no match

df['district'] = df['district'].apply(map_district)



In [937]:
# map district NANs from known origin station:

station_district_mapping = {
    'Central and Western': ('Central', 'Admiralty', 'Kennedy Town'),
    'Wan Chai': ('Wan Chai', 'Causeway Bay'),
    'Yau Tsim Mong': ('Mong Kok', 'Tsim Sha Tsui'),
    'Sha Tin': ('Sha Tin',),
    'Tsuen Wan': ('Tsuen Wan',),
    'Eastern': ('North Point',)
}


def fill_district_from_station(df, station_col='origin_station', district_col='district'):
    # Build reverse lookup: station -> district
    station_to_district = {
        station.lower(): district
        for district, stations in station_district_mapping.items()
        for station in stations
    }
    
    # Fill nulls with station lookup
    df[district_col] = df[district_col].fillna(
        df[station_col].str.lower().str.strip().map(station_to_district)
    )
    return df

# Usage
df = fill_district_from_station(df)

df['district'].value_counts(dropna=False)

district
Central and Western    543
Yau Tsim Mong          431
Wan Chai               405
Sha Tin                281
Tsuen Wan              193
NaN                    127
Eastern                113
Name: count, dtype: int64

In [938]:
df[['origin_station','destination_station','district']].sample(30)

,origin_station,destination_station,district
1424,Causeway Bay,Central,Wan Chai
585,Central,Tsuen Wan,Central and Western
1030,Causeway Bay,Mong Kok,Wan Chai
1377,Kennedy Town,Central,Central and Western
1723,Kennedy Town,Mong Kok,Central and Western
1916,NaN,NaN,NaN
910,Causeway Bay,Central,Wan Chai
527,Kennedy Town,Tsim Sha Tsui,Central and Western
1020,Tsuen Wan,Wan Chai,Tsuen Wan
1888,North Point,Tsim Sha Tsui,Sha Tin


Cleaning Day of Week

In [939]:
df['day_of_week'] = df['day_of_week'].str.title()
df['day_of_week'].value_counts(dropna=False)

day_of_week
Fri    366
Thu    329
Wed    324
Mon    308
Tue    290
Sat    281
Sun    195
Name: count, dtype: int64

In [940]:
df

,origin_station,destination_station,district,encoded_transport,day_of_week,is_holiday,weather_condition,country_code
0,Sha Tin,Tsuen Wan,Sha Tin,tram_loc_std_citybus(src=endcap)(batch=L2),Fri,0,NaN,UNK
1,Tsuen Wan,Kennedy Town,Tsuen Wan,type=tram-xhbr;mode=loc;tier=missing;op=unknown,Thu,0,NaN,852
2,Causeway Bay,Wan Chai,Wan Chai,[tram][run=loc][tier=std][op=citybus][sys=L2],Wed,0,NaN,territory-hk
3,Admiralty,Causeway Bay,Central and Western,[bus-ngt][run=exp][tier=std][op=citybus][sys=L2],Thu,0,wx_unknown,legacy
4,North Point,Central,Sha Tin,L2::bus::exp::std::citybus::tail,Sat,1,NaN,legacy
...,...,...,...,...,...,...,...,...
2088,NaN,NaN,Sha Tin,maintenance_shift,Sun,0,SUNNY,TST
2089,Wan Chai,Central,Yau Tsim Mong,feed=OPS|kind=tram-ngt|run=exp|tier=std|op=cit...,Sat,0,SUNNY,cc=852|OPS
2090,Sha Tin,Tsuen Wan,Wan Chai,ferry__MISS__ferryhk,Tue,0,heavy-rn,legacy-null
2091,Mong Kok,Wan Chai,Yau Tsim Mong,L2::tram::exp::std::citybus::tail,Mon,0,heavy-rn,territory-hk


In [941]:
clean.SUBMISSION_FEATURES

['origin_station',
 'destination_station',
 'district',
 'transport_type',
 'transport_detail',
 'mode',
 'service_level',
 'operator',
 'day_of_week',
 'is_holiday',
 'weather_condition',
 'country_code']

In [942]:
# Parse encoded_transport into 5 columns
_SPECIAL_NO_TRANSPORT = {
    'maintenance_shift', 'admin_move', 'staff_shuttle', 'test_run',
    'unk', 'encoded_transport'
}

def parse_encoded_transport(s):
    if not isinstance(s, str):
        return {'transport_type': pd.NA, 'transport_detail': pd.NA,
                'mode': pd.NA, 'service_level': pd.NA, 'operator': pd.NA}
    v = s.strip().lower()
    if v in _SPECIAL_NO_TRANSPORT or not v:
        return {'transport_type': pd.NA, 'transport_detail': pd.NA,
                'mode': pd.NA, 'service_level': pd.NA, 'operator': pd.NA}

    # transport_type
    if re.search(r'\btram\b', v):
        transport_type = 'tram'
    elif re.search(r'\bferry\b', v):
        transport_type = 'ferry'
    elif re.search(r'\bbus\b', v):
        transport_type = 'bus'
    else:
        transport_type = pd.NA

    # transport_detail
    m = re.search(r'(tram|ferry|bus)[-]?(apt|ngt|night|xhbr)', v)
    if m:
        detail_map = {'apt': 'airport', 'ngt': 'night', 'night': 'night', 'xhbr': 'crossharbour'}
        transport_detail = detail_map.get(m.group(2), 'general')
    else:
        transport_detail = 'general'

    # mode
    m = re.search(r'\b(?:run|mode)[=:](\w+)', v)
    if m:
        mode_map = {'loc': 'local', 'exp': 'express'}
        mode = mode_map.get(m.group(1), pd.NA)
    else:
        m = re.search(r'(tram|ferry|bus)[-]?\w*[:_](\w+)[:_]', v)
        if m and m.group(2) in ('loc', 'exp'):
            mode = 'local' if m.group(2) == 'loc' else 'express'
        else:
            mode = pd.NA

    # service_level
    m = re.search(r'\b(?:tier|svc)[=:](\w+)', v)
    if m:
        sl_map = {'std': 'standard', 'prem': 'premium', 'missing': pd.NA}
        service_level = sl_map.get(m.group(1), pd.NA)
    else:
        m = re.search(r'(tram|ferry|bus)[-]?\w*[:_](\w+)[:_]', v)
        if m and m.group(2) in ('std', 'prem'):
            service_level = 'standard' if m.group(2) == 'std' else 'premium'
        else:
            service_level = pd.NA

    # operator
    m = re.search(r'\b(?:op|owner)[=:](\w+)', v)
    if m:
        operator = m.group(1)
    else:
        m = re.search(r'(?<![|:])(\w+)[-|_]?(tail|delta|v2|\?|\?\?|c)$', v)
        if m:
            operator = m.group(1)
        else:
            parts = re.split(r'[:_|\[\];=,\.\-]+', v)
            parts = [p for p in parts if p not in ('', '??','unk', 'unknown', 'l2)', 'tail','ops', 'srcfeed', 'l2', 'amb', 'legacy', 'feed', 'type', 'mode', 'run', 'tier', 'svc', 'op', 'owner', 'sys', 'batch', 'src', 'kind')]
            operator = parts[-1] if parts else pd.NA
            if operator in ('std', 'prem', 'loc', 'exp', 'unk', ''):
                operator = pd.NA

    # Handle concatenated patterns like buslocstdkowloon -> extract operator at end
    m = re.search(r'(citybus|kowloon|ferryhk)$', str(operator))
    if m:
        operator = m.group(1)

    # Fix truncated operators (missing first letter)
    if operator == 'itybus':
        operator = 'citybus'
    elif operator == 'owloon':
        operator = 'kowloon'
    elif operator == 'erryhk':
        operator = 'ferryhk'

    # Normalize garbage/invalid operators to NaN
    invalid_operators = {'unknown', 'delta', 'endcap)(batch', ''}
    if operator in invalid_operators:
        operator = pd.NA

    return {'transport_type': transport_type, 'transport_detail': transport_detail,
            'mode': mode, 'service_level': service_level, 'operator': operator}

parsed = df['encoded_transport'].apply(parse_encoded_transport).apply(pd.Series)
df = pd.concat([df, parsed], axis=1)

df[['encoded_transport', 'transport_type',
 'transport_detail',
 'mode',
 'service_level',
 'operator']].sample(20)

,encoded_transport,transport_type,transport_detail,mode,service_level,operator
873,type:tram|mode:loc|svc:std|owner:citybus|feed=OPS,tram,general,local,standard,citybus
1781,maintenance_shift,NaN,NaN,NaN,NaN,NaN
856,type=tram;mode=loc;tier=missing;op=unknown,tram,general,local,NaN,NaN
474,OPS::bus|loc|std|kowloon|v2,bus,general,NaN,NaN,kowloon
481,UNK::tram::loc::citybus,tram,general,NaN,NaN,citybus
1580,srcfeed::tram::loc::std::citybus::delta,tram,general,NaN,NaN,NaN
489,amb::bus:loc:prem:citybus:??,bus,general,local,NaN,citybus
1850,kind=bus-apt|kind=alt|run=?|tier=?|op=kowloon,bus,airport,NaN,NaN,kowloon
178,ferry__MISS__ferryhk,NaN,general,NaN,NaN,ferryhk
926,type=tram;mode=loc;tier=missing;op=unknown,tram,general,local,NaN,NaN


In [943]:
df['operator'].value_counts(dropna=False).head(30)

operator
citybus    802
kowloon    599
NaN        438
ferryhk    254
Name: count, dtype: int64